# Desafio do Inteli Academy
## Por Marco Ruas Sales Peixoto
### Ex - Futuro Vice-Presidente da Liga

Neste notebook, vai ser documentado o desafio de machine learning para resolver o case do processo seletivo da Liga.

As etapas feitas no processo foram as seguintes:

1 - `Exploração de dados` – análise inicial das variáveis, identificação de padrões e possíveis inconsistências.
Pré-processamento – tratamento de valores ausentes, normalização, engenharia de features e outras transformações necessárias.

2 - `Geração de insights e formulação de hipóteses` – interpretação dos dados e dos fatores mais relevantes para a previsão de churn.

3 - `Treinamento e teste de modelos` – experimentação com diferentes algoritmos para encontrar o melhor desempenho.

4 - `Avaliação e comparação de modelos` – análise de métricas de performance e justificativa da escolha do modelo final.

5 - `Geração do arquivo de previsões` – ao final do processo, seu notebook deve exportar um CSV chamado "resultado_nome_sobrenome.csv" contendo as previsões para o dataset de teste ("desafio.csv").


Uma nota: Decidi modularizar o código dividindo em funções, facilitando a reutilização em outra etapa, e também fiz várias plotagens de mapas para entender mais as features e em muitas vezes não utilizei, mas serviu de aprendizado.

In [ ]:
# Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc


# Exploração de Dados

Aqui eu tentei brincar e lembrar de como fazer uma exploração bem feita, admito que a maioria dos dados que li entendi foi nada, mas depois de uns prompts e lê novamente, consegui entender e definir futuramente features.


In [ ]:
# Carregar os dados de treino
df_train = pd.read_csv('dados_clientes.csv')

# Carregar os dados de teste
df_test = pd.read_csv('desafio.csv')

# Verificar as dimensões dos dados
print("Dimensões do conjunto de treino:", df_train.shape)
print("Dimensões do conjunto de teste:", df_test.shape)

# Verificar as primeiras linhas do conjunto de treino
df_train.head()

In [ ]:
# Informações sobre os dados de treino
df_train.info()

# Resumo estatístico dos dados numéricos
df_train.describe()

# Verificar valores ausentes
print("Valores ausentes no conjunto de treino:")
print(df_train.isnull().sum())

# Verificar distribuição da variável alvo (churn)
print("\nDistribuição da variável alvo (churn):")
print(df_train['churn'].value_counts())
print("Porcentagem de churn:", df_train['churn'].mean() * 100, "%")

In [ ]:
numeric_cols = df_train.select_dtypes(include=['int64', 'float64']).columns
correlation = df_train[numeric_cols].corr()

# Plotar mapa de calor da correlação
plt.figure(figsize=(14, 10))
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# Função para plotar histograma por status de churn
def plot_histogram_by_churn(df, column):
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x=column, hue='churn', multiple='stack', bins=20)
    plt.title(f'Distribuição de {column} por Status de Churn')
    plt.xlabel(column)
    plt.ylabel('Contagem')
    plt.show()

# Variáveis numéricas para analisar
numeric_features = ['idade', 'tempo_como_cliente', 'suporte_contatado',
                    'chamados_abertos', 'tempo_medio_atencao', 'reclamacoes',
                    'atrasos_pagamento', 'valor_mensal', 'total_gasto']

# Plotar histogramas para as variáveis numéricas
for feature in numeric_features:
    if feature in df_train.columns:
        plot_histogram_by_churn(df_train, feature)

# Análise de variáveis categóricas
categorical_features = ['genero', 'estado_civil', 'tipo_contrato',
                        'forma_pagamento', 'renda_faixa', 'servicos_assinados']

# Plotar gráficos de barras para variáveis categóricas
for feature in categorical_features:
    if feature in df_train.columns:
        plt.figure(figsize=(10, 6))
        df_grouped = df_train.groupby([feature, 'churn']).size().unstack()
        df_grouped_percentage = df_grouped.div(df_grouped.sum(axis=1), axis=0) * 100
        df_grouped_percentage.plot(kind='bar', stacked=False)
        plt.title(f'Taxa de Churn por {feature}')
        plt.xlabel(feature)
        plt.ylabel('Porcentagem de Churn (%)')
        plt.legend(['Não Cancelou (0)', 'Cancelou (1)'])
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
# Criar features derivadas que possam ser úteis
def create_features(df):
    df_copy = df.copy()

    # Relação entre reclamações e tempo como cliente
    if 'reclamacoes' in df_copy.columns and 'tempo_como_cliente' in df_copy.columns:
        df_copy['reclamacoes_por_tempo'] = df_copy['reclamacoes'] / (df_copy['tempo_como_cliente'] + 1)

    # Relação entre suporte contatado e chamados abertos
    if 'suporte_contatado' in df_copy.columns and 'chamados_abertos' in df_copy.columns:
        df_copy['taxa_resolucao'] = df_copy['chamados_abertos'] / (df_copy['suporte_contatado'] + 1)

    # Relação entre valor gasto e tempo como cliente
    if 'total_gasto' in df_copy.columns and 'tempo_como_cliente' in df_copy.columns:
        df_copy['gasto_medio_mensal'] = df_copy['total_gasto'] / (df_copy['tempo_como_cliente'] + 1)

    # Indicador de problemas recentes
    if 'reclamacoes' in df_copy.columns and 'atrasos_pagamento' in df_copy.columns:
        df_copy['indicador_problemas'] = df_copy['reclamacoes'] + df_copy['atrasos_pagamento']

    # Categorizar tempo como cliente
    if 'tempo_como_cliente' in df_copy.columns:
        df_copy['faixa_tempo_cliente'] = pd.cut(
            df_copy['tempo_como_cliente'],
            bins=[0, 12, 24, 36, 60, float('inf')],
            labels=['<1 ano', '1-2 anos', '2-3 anos', '3-5 anos', '>5 anos']
        )

    return df_copy

# Aplicar engenharia de features
df_train_featured = create_features(df_train)
df_test_featured = create_features(df_test)

# Verificar as novas colunas
print("Novas features criadas:")
new_columns = list(set(df_train_featured.columns) - set(df_train.columns))
print(new_columns)

# Pré-processamento
Nesta etapa que começa a preparar o terreno para poder treinar o modelo, definindo as variáveis e o alvo.

In [ ]:
# Separar variáveis independentes e variável alvo
X_train = df_train_featured.drop('churn', axis=1)
y_train = df_train_featured['churn']

# Identificar colunas categóricas e numéricas
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Excluir 'id_cliente' das colunas a serem usadas
if 'id_cliente' in numerical_cols:
    numerical_cols.remove('id_cliente')
if 'id_cliente' in categorical_cols:
    categorical_cols.remove('id_cliente')

print("Colunas numéricas:", numerical_cols)
print("Colunas categóricas:", categorical_cols)

# Definir transformadores para os diferentes tipos de colunas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinar transformadores usando ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Dividir o conjunto de treino em treino e validação
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print("Tamanho do conjunto de treino:", X_train_split.shape)
print("Tamanho do conjunto de validação:", X_val.shape)

# Treinamento dos Modelos

Eu escolhi fazer com Random Forest e Xgboost, mas vi pessoas tendo bons resultados com regressão logística, então porque não verificar também né?

(Não adiantou nada)

In [ ]:
# Definir modelos a serem testados
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42)
}

# Função para avaliar modelos
def evaluate_model(name, model, X_train, X_val, y_train, y_val, preprocessor):
    # Criar pipeline com preprocessamento e modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Treinar o modelo
    pipeline.fit(X_train, y_train)

    # Fazer previsões
    y_pred = pipeline.predict(X_val)
    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]

    # Calcular métricas
    accuracy = accuracy_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_pred_proba)

    # Exibir métricas
    print(f"\n{name} - Resultados:")
    print(f"Acurácia: {accuracy:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")

    # Matriz de confusão
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Matriz de Confusão - {name}')
    plt.ylabel('Real')
    plt.xlabel('Previsto')
    plt.show()

    # Relatório de classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_val, y_pred))

    return pipeline, accuracy, roc_auc

# Avaliar cada modelo
results = {}
for name, model in models.items():
    print(f"\n{'=' * 50}")
    print(f"Avaliando {name}")
    print(f"{'=' * 50}")

    pipeline, accuracy, roc_auc = evaluate_model(
        name, model, X_train_split, X_val, y_train_split, y_val, preprocessor
    )

    results[name] = {
        'pipeline': pipeline,
        'accuracy': accuracy,
        'roc_auc': roc_auc
    }

# Comparar modelos
model_comparison = pd.DataFrame({
    'Modelo': list(results.keys()),
    'Acurácia': [results[model]['accuracy'] for model in results],
    'ROC AUC': [results[model]['roc_auc'] for model in results]
})

print("\nComparação dos Modelos:")
print(model_comparison.sort_values('Acurácia', ascending=False))

## Código pra fazer um charme e verificar qual modelo é melhor de acordo com a acurácia

In [ ]:
# Selecionar o melhor modelo com base na acurácia
best_model_name = model_comparison.sort_values('Acurácia', ascending=False).iloc[0]['Modelo']
print(f"\nOtimizando o melhor modelo: {best_model_name}")

# Definir parâmetros de otimização para cada modelo
param_grids = {
    'Logistic Regression': {
        'classifier__C': [0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__solver': ['liblinear', 'saga']
    },
    'Random Forest': {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20, 30],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4]
    },
    'XGBoost': {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 5, 7],
        'classifier__learning_rate': [0.01, 0.1, 0.2],
        'classifier__subsample': [0.8, 0.9, 1.0],
        'classifier__colsample_bytree': [0.8, 0.9, 1.0]
    }
}

# Criar pipeline para o melhor modelo
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', models[best_model_name])
])

# Configurar busca em grade
grid_search = GridSearchCV(
    best_pipeline,
    param_grids[best_model_name],
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Executar busca em grade
grid_search.fit(X_train, y_train)

# Exibir melhores parâmetros
print("\nMelhores parâmetros encontrados:")
print(grid_search.best_params_)

# Avaliar modelo otimizado
best_model = grid_search.best_estimator_
y_val_pred = best_model.predict(X_val)
best_accuracy = accuracy_score(y_val, y_val_pred)

print(f"\nAcurácia do modelo otimizado: {best_accuracy:.4f}")
print("\nRelatório de classificação do modelo otimizado:")
print(classification_report(y_val, y_val_pred))

# Codificação das features

In [ ]:
# Função para extrair os nomes das features após one-hot encoding
def get_feature_names(column_transformer):
    output_features = []

    for name, pipe, features in column_transformer.transformers_:
        if name != 'remainder':
            if hasattr(pipe.named_steps['onehot'], 'get_feature_names_out'):
                if isinstance(features, list):
                    cat_features = pipe.named_steps['onehot'].get_feature_names_out(features)
                    output_features.extend(cat_features)
                else:
                    cat_features = pipe.named_steps['onehot'].get_feature_names_out([features])
                    output_features.extend(cat_features)
            else:
                output_features.extend(features)

    return output_features

# Extrair importância das features (para Random Forest ou XGBoost)
if best_model_name in ['Random Forest', 'XGBoost']:
    # Obter os nomes das features após transformação
    preprocessor_fitted = best_model.named_steps['preprocessor']
    feature_names = numerical_cols.copy()

    # Adicionar nomes de features categóricas após one-hot encoding
    for name, transformer, column in preprocessor_fitted.transformers_:
        if name == 'cat':
            cat_features = transformer.named_steps['onehot'].get_feature_names_out(categorical_cols)
            feature_names.extend(cat_features)

    # Extrair importância das features
    feature_importances = best_model.named_steps['classifier'].feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importances
    }).sort_values('Importance', ascending=False)

    # Plotar importância das features
    plt.figure(figsize=(12, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(20))
    plt.title('Top 20 Features Mais Importantes')
    plt.tight_layout()
    plt.show()

    print("\nTop 10 Features Mais Importantes:")
    print(feature_importance_df.head(10))

# Extração do csv final

In [ ]:
# Preparar dados de teste
X_test = df_test_featured.copy()
if 'churn' in X_test.columns:
    X_test = X_test.drop('churn', axis=1)

# Fazer predições no conjunto de teste
test_predictions = best_model.predict(X_test)

# Criar DataFrame com os resultados
results_df = pd.DataFrame({
    'Id': X_test['id_cliente'] if 'id_cliente' in X_test.columns else X_test.index,
    'Target': test_predictions
})

# Salvar resultados em CSV
results_df.to_csv('resultado_marco_peixoto.csv', index=False)


# Novo Teste
Então a partir de aqui foi full na base do Gemini para tentar atingir a acurácia de 90% Onde consegui a proeza de atingir 94%.

Observação: Depois de uma nova análise, percebi (obviamente) que isso estava mais overfittado que tudo, então fiz um novo ajuste no código abaixo e tentei novamente o teste, e reduziu a acurácia, mas não consegui gerar o csv pois o modelo demora no mínimo 3 horas para rodar.







In [ ]:
# -*- coding: utf-8 -*-
"""
Notebook Otimizado para Previsão de Churn - TelecomPlus

Autor: Claude (com otimizações)
"""

# -----------------------------------------------------
# 1. Importando bibliotecas necessárias
# -----------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import ast
from collections import Counter

# Pré-processamento e Engenharia de Features
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier # Descomente se quiser testar LightGBM
# from catboost import CatBoostClassifier # Descomente se quiser testar CatBoost

# Métricas
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve, auc, f1_score, make_scorer
)

# Para lidar com desbalanceamento (se usar SMOTE)
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None) # Mostrar todas as colunas

print("Bibliotecas importadas com sucesso!")

# -----------------------------------------------------
# 2. Carregando os dados
# -----------------------------------------------------
try:
    # Usando os nomes de arquivo fornecidos
    df_train_raw = pd.read_csv('dados_clientes.csv')
    df_test_raw = pd.read_csv('desafio.csv')
    print("Arquivos CSV carregados com sucesso.")
    print("Dimensões do conjunto de treino (raw):", df_train_raw.shape)
    print("Dimensões do conjunto de teste (raw):", df_test_raw.shape)
except FileNotFoundError:
    print("Erro: Verifique se os arquivos 'dadosclientesnovo.csv' e 'desafionovo.csv' estão no diretório correto.")
    exit()

# Verificar IDs únicos (importante se id_cliente for usado como índice)
if df_train_raw['id_cliente'].nunique() != len(df_train_raw):
    print("Aviso: IDs de cliente duplicados no conjunto de treino.")
if df_test_raw['id_cliente'].nunique() != len(df_test_raw):
    print("Aviso: IDs de cliente duplicados no conjunto de teste.")

# Guardar IDs do teste para o final
test_ids = df_test_raw['id_cliente']

# -----------------------------------------------------
# 3. Exploração Inicial e Pré-processamento Combinado
# -----------------------------------------------------

# Renomear coluna de tempo médio para evitar problemas de nome
if 'tempo_medio_atendimento' in df_train_raw.columns:
    df_train_raw.rename(columns={'tempo_medio_atendimento': 'tempo_medio_atencao'}, inplace=True)
if 'tempo_medio_atendimento' in df_test_raw.columns:
    df_test_raw.rename(columns={'tempo_medio_atendimento': 'tempo_medio_atencao'}, inplace=True)

# -- Função para Parsear Lista de Produtos --
def parse_product_list(product_string):
    """Parseia a string da lista de produtos, lidando com formatos como ['A' 'B']."""
    if not isinstance(product_string, str) or product_string == '[]':
        return []
    try:
        # Tenta usar regex para extrair itens entre aspas simples
        products = re.findall(r"'([^']+)'", product_string)
        return [p.strip() for p in products if p.strip()]
    except Exception:
        return [] # Retorna lista vazia em caso de erro

# -- Função de Pré-processamento e Engenharia de Features --
def preprocess_and_engineer_features(df, fit_mlb=False, mlb_instance=None, product_columns=None):
    """Aplica pré-processamento e engenharia de features."""
    df_copy = df.copy()

    # 1. Parsear produtos_assinados
    df_copy['produtos_lista'] = df_copy['produtos_assinados'].apply(parse_product_list)

    # 2. Criar Dummies para Produtos com MultiLabelBinarizer
    if fit_mlb:
        mlb = MultiLabelBinarizer()
        product_dummies = pd.DataFrame(mlb.fit_transform(df_copy['produtos_lista']),
                                       columns=[f'prod_{cls}' for cls in mlb.classes_],
                                       index=df_copy.index)
        product_columns = product_dummies.columns.tolist() # Salva as colunas criadas
        mlb_instance = mlb # Salva a instância treinada
    else:
        if mlb_instance is None or product_columns is None:
             raise ValueError("Instância MLB e colunas de produtos são necessárias quando fit_mlb=False.")
        # Garante que o test set tenha as mesmas colunas de produtos que o train set
        product_dummies_test = pd.DataFrame(mlb_instance.transform(df_copy['produtos_lista']),
                                            columns=product_columns, # Usa as colunas do treino
                                            index=df_copy.index)
        # Adiciona colunas faltantes se houver produtos novos (preenche com 0) - Raro se MLB foi bem treinado
        missing_cols = set(product_columns) - set(product_dummies_test.columns)
        for c in missing_cols:
            product_dummies_test[c] = 0
        product_dummies = product_dummies_test[product_columns] # Reordena e seleciona

    df_copy = pd.concat([df_copy, product_dummies], axis=1)

    # 3. Codificar renda_faixa (Ordinal)
    # Definindo a ordem correta das faixas
    renda_order = ['0-1000', '1000-5000', '5000-10000', '10000-50000', '50000-100000']
    # Tratar NaNs ou valores inesperados antes de mapear
    df_copy['renda_faixa'].fillna('Desconhecido', inplace=True) # Ou use a moda
    # Adiciona categorias não vistas no treino à lista para o encoder não falhar
    unique_renda = df_copy['renda_faixa'].unique().tolist()
    all_renda_categories = sorted(list(set(renda_order + unique_renda))) + ['Desconhecido'] if 'Desconhecido' not in unique_renda else sorted(list(set(renda_order + unique_renda)))


    ordinal_encoder = OrdinalEncoder(categories=[all_renda_categories], handle_unknown='use_encoded_value', unknown_value=np.nan) # Lida com categorias inesperadas
    # Aplicar o encoder (reshape é necessário para uma única feature)
    df_copy['renda_ordinal'] = ordinal_encoder.fit_transform(df_copy[['renda_faixa']])
     # Se gerou NaN (categoria desconhecida), imputar depois ou aqui (ex: com -1 ou a média)
    df_copy['renda_ordinal'].fillna(-1, inplace=True) # Usando -1 para desconhecido


    # 4. Criar features derivadas (Engenharia de Features Original)
    # Adicionando tratamento para divisão por zero ou nulos
    df_copy['tempo_como_cliente_safe'] = df_copy['tempo_como_cliente'].replace(0, 1) # Evita divisão por zero
    df_copy['suporte_contatado_safe'] = df_copy['suporte_contatado'].replace(0, 1)

    df_copy['reclamacoes_por_tempo'] = df_copy['reclamacoes'].fillna(0) / df_copy['tempo_como_cliente_safe']
    df_copy['taxa_resolucao'] = df_copy['chamados_abertos'].fillna(0) / (df_copy['suporte_contatado'].fillna(0) + 1) # +1 evita div por zero
    df_copy['gasto_medio_mensal_calc'] = df_copy['total_gasto'].fillna(0) / df_copy['tempo_como_cliente_safe']
    df_copy['indicador_problemas'] = df_copy['reclamacoes'].fillna(0) + df_copy['atrasos_pagamento'].fillna(0)

    # Cuidado com valores potencialmente infinitos ou muito grandes
    df_copy.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 5. Remover colunas originais/intermediárias e identificar tipos
    cols_to_drop = ['id_cliente', 'produtos_assinados', 'produtos_lista', 'renda_faixa', 'tempo_como_cliente_safe', 'suporte_contatado_safe']
    df_processed = df_copy.drop(columns=cols_to_drop, errors='ignore')

    numerical_cols = df_processed.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = df_processed.select_dtypes(include=['object', 'category']).columns.tolist()

    # Remover a variável alvo das features se ela existir
    if 'churn' in numerical_cols:
        numerical_cols.remove('churn')
    if 'churn' in categorical_cols:
        categorical_cols.remove('churn')

    # Remover colunas dummy de produtos das categóricas (já estão nas numéricas)
    categorical_cols = [col for col in categorical_cols if not col.startswith('prod_')]

    return df_processed, numerical_cols, categorical_cols, mlb_instance, product_columns

# -- Aplicar a função --
print("\nIniciando pré-processamento e engenharia de features...")
df_train_processed, numerical_cols, categorical_cols, mlb_fitted, product_cols = preprocess_and_engineer_features(df_train_raw, fit_mlb=True)
df_test_processed, _, _, _, _ = preprocess_and_engineer_features(df_test_raw, fit_mlb=False, mlb_instance=mlb_fitted, product_columns=product_cols)

print("Pré-processamento concluído.")
print("Colunas numéricas identificadas:", numerical_cols)
print("Colunas categóricas identificadas:", categorical_cols)
print("Dimensões do treino processado:", df_train_processed.shape)
print("Dimensões do teste processado:", df_test_processed.shape)

# Verificar se a coluna 'churn' existe no df_train_processed
if 'churn' not in df_train_processed.columns:
    print("Erro: Coluna 'churn' não encontrada no DataFrame de treino processado.")
    exit()

# -----------------------------------------------------
# 4. Preparação para Modelagem
# -----------------------------------------------------
X = df_train_processed.drop('churn', axis=1)
y = df_train_processed['churn']

# Assegurar que X_test tenha as mesmas colunas que X (exceto 'churn')
# Colunas que estão em X mas não em X_test
missing_cols_test = set(X.columns) - set(df_test_processed.columns)
for c in missing_cols_test:
    df_test_processed[c] = 0 # Adiciona colunas faltantes com valor 0
# Colunas que estão em X_test mas não em X (raro, mas possível)
extra_cols_test = set(df_test_processed.columns) - set(X.columns)
df_test_processed = df_test_processed.drop(columns=list(extra_cols_test))

# Garantir a mesma ordem de colunas
X_test = df_test_processed[X.columns]

print("\nConjuntos X e y criados.")
print("Dimensões de X (treino):", X.shape)
print("Dimensões de y (treino):", y.shape)
print("Dimensões de X_test:", X_test.shape)

# Verificar distribuição da variável alvo
churn_counts = y.value_counts()
churn_percentage = y.mean() * 100
print(f"\nDistribuição da variável alvo (churn): \n{churn_counts}")
print(f"Porcentagem de churn: {churn_percentage:.2f}%")

# Calcular scale_pos_weight para XGBoost
neg_count = churn_counts[0]
pos_count = churn_counts[1]
scale_pos_weight_value = neg_count / pos_count
print(f"Valor calculado para scale_pos_weight: {scale_pos_weight_value:.4f}")

# Dividir dados para validação interna (se não usar cross-validation no GridSearch)
# X_train_split, X_val, y_train_split, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
# print("\nDados divididos em treino/validação (80/20).")
# print("Tamanho do conjunto de treino split:", X_train_split.shape)
# print("Tamanho do conjunto de validação:", X_val.shape)

# -----------------------------------------------------
# 5. Definir Pipeline de Pré-processamento Final
# -----------------------------------------------------
# Ajustar as listas de colunas caso alguma numérica/categórica tenha sido removida/adicionada
final_numerical_cols = [col for col in numerical_cols if col in X.columns]
final_categorical_cols = [col for col in categorical_cols if col in X.columns]

# Usar SimpleImputer com estratégia 'median' para numéricas e 'most_frequent' para categóricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Mediana é mais robusta a outliers
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Imputar com a moda
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # handle_unknown='ignore' é crucial
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, final_numerical_cols),
        ('cat', categorical_transformer, final_categorical_cols)
    ],
    remainder='passthrough' # Mantém colunas não especificadas (se houver, cuidado!)
)
print("\nPipeline de pré-processamento definido.")

# -----------------------------------------------------
# 6. Otimização do Modelo XGBoost com GridSearchCV
# -----------------------------------------------------
print("\nIniciando otimização do XGBoost com GridSearchCV...")

# Criar o pipeline completo: pré-processador + classificador XGBoost
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42,
                                 objective='binary:logistic', # Para classificação binária
                                 eval_metric='logloss', # Métrica de avaliação interna
                                 use_label_encoder=False)) # Evita warning
])

# Definir o espaço de busca de hiperparâmetros (ajustado)
param_grid_xgb = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 0.9],      # Amostragem de linhas
    'classifier__colsample_bytree': [0.7, 0.8, 0.9],# Amostragem de colunas por árvore
    'classifier__gamma': [0, 0.1, 0.5],             # Regularização (min loss reduction)
    'classifier__scale_pos_weight': [scale_pos_weight_value] # Usar valor calculado
    #'classifier__reg_alpha': [0, 0.01, 0.1], # L1 Regularization (opcional)
    #'classifier__reg_lambda': [0.1, 1, 10],  # L2 Regularization (opcional)
}

# Configurar a validação cruzada estratificada
# CV mais rápido para demonstração, aumente para 5 ou 10 para mais robustez
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Configurar GridSearchCV
# Usando roc_auc como métrica principal, mas reportando outras
scoring = {
    'AUC': 'roc_auc',
    'Accuracy': 'accuracy',
    'F1': 'f1'
}

grid_search = GridSearchCV(
    xgb_pipeline,
    param_grid=param_grid_xgb,
    cv=kfold,
    scoring=scoring,
    refit='AUC', # Escolhe o melhor modelo baseado no AUC ROC
    n_jobs=-1,    # Usar todos os cores disponíveis
    verbose=1     # Mostrar progresso
)

# Executar a busca em grade (usando todo o conjunto de treino X, y)
grid_search.fit(X, y)

# Exibir os melhores resultados
print("\nOtimização concluída!")
print(f"Melhor pontuação AUC ROC (CV): {grid_search.best_score_:.4f}")
print("Melhores parâmetros encontrados:")
print(grid_search.best_params_)

# Guardar o melhor modelo encontrado
best_model = grid_search.best_estimator_

# Avaliar o melhor modelo no conjunto de treino completo (para referência, pode haver overfitting)
y_train_pred = best_model.predict(X)
y_train_pred_proba = best_model.predict_proba(X)[:, 1]

print("\n--- Avaliação do Melhor Modelo no Conjunto de Treino Completo ---")
print(f"Acurácia (Treino): {accuracy_score(y, y_train_pred):.4f}")
print(f"ROC AUC (Treino): {roc_auc_score(y, y_train_pred_proba):.4f}")
print(f"F1 Score (Treino): {f1_score(y, y_train_pred):.4f}")
print("Relatório de Classificação (Treino):")
print(classification_report(y, y_train_pred))
print("Matriz de Confusão (Treino):")
cm_train = confusion_matrix(y, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusão (Treino)')
plt.ylabel('Real')
plt.xlabel('Previsto')
plt.show()


# -----------------------------------------------------
# 7. Análise de Importância das Features
# -----------------------------------------------------
try:
    # Obter nomes das features após pré-processamento
    # Etapa 1: Nomes das colunas numéricas (já transformadas)
    num_features = final_numerical_cols

    # Etapa 2: Nomes das colunas categóricas após OneHotEncoding
    cat_features = best_model.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(final_categorical_cols)

    # Etapa 3: Combinar nomes
    all_feature_names = list(num_features) + list(cat_features)

    # Obter importâncias do classificador XGBoost treinado
    importances = best_model.named_steps['classifier'].feature_importances_

    # Criar DataFrame de importância
    feature_importance_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    # Plotar as Top N features
    N = 25 # Número de features a mostrar
    plt.figure(figsize=(12, N * 0.3)) # Ajustar tamanho
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(N))
    plt.title(f'Top {N} Features Mais Importantes (XGBoost)')
    plt.tight_layout()
    plt.show()

    print(f"\nTop 10 Features Mais Importantes:\n{feature_importance_df.head(10)}")

except Exception as e:
    print(f"\nErro ao gerar importância das features: {e}")
    print("Isso pode acontecer se o pré-processador não tiver sido ajustado corretamente ou o modelo não for baseado em árvore.")


# -----------------------------------------------------
# 8. Predições Finais no Conjunto de Teste
# -----------------------------------------------------
print("\nRealizando predições no conjunto de teste...")

# Fazer predições no conjunto de teste já pré-processado (X_test)
test_predictions = best_model.predict(X_test)
# test_predictions_proba = best_model.predict_proba(X_test)[:, 1] # Se precisar das probabilidades

print("Predições concluídas.")
print("Distribuição das previsões no teste:")
print(pd.Series(test_predictions).value_counts())

# -----------------------------------------------------
# 9. Geração do Arquivo de Resultado
# -----------------------------------------------------
# Criar DataFrame com os resultados no formato esperado
results_df = pd.DataFrame({
    'id_cliente': test_ids, # Usar os IDs guardados no início
    'churn': test_predictions # Coluna alvo geralmente chamada 'churn' ou 'Target'
})

# Definir nome do arquivo de saída
output_filename = 'resultado_claude_otimizado.csv' # **RENOMEIE SE NECESSÁRIO**

# Salvar resultados em CSV sem o índice
results_df.to_csv(output_filename, index=False)

print(f"\nArquivo de resultados '{output_filename}' gerado com sucesso!")
print(results_df.head())

# -----------------------------------------------------
# 10. Conclusões (Placeholder)
# -----------------------------------------------------
print("\n--- Fim do Processo ---")
print(f"O modelo final (XGBoost otimizado) foi treinado e as previsões foram salvas.")
print("Verifique o arquivo CSV gerado e analise a importância das features para insights.")
print(f"A melhor pontuação ROC AUC obtida na validação cruzada foi: {grid_search.best_score_:.4f}")
print("Avalie a performance final na plataforma do desafio (se aplicável).")
# Adicione suas conclusões aqui com base nos resultados e na importância das features.

Bibliotecas importadas com sucesso!
Arquivos CSV carregados com sucesso.
Dimensões do conjunto de treino (raw): (98872, 18)
Dimensões do conjunto de teste (raw): (5000, 17)

Iniciando pré-processamento e engenharia de features...
Pré-processamento concluído.
Colunas numéricas identificadas: ['idade', 'tempo_como_cliente', 'suporte_contatado', 'chamados_abertos', 'tempo_medio_atencao', 'reclamacoes', 'atrasos_pagamento', 'servicos_assinados', 'valor_mensal', 'total_gasto', 'prod_Produto A', 'prod_Produto B', 'prod_Produto C', 'prod_Produto D', 'prod_Produto E', 'prod_Produto F', 'renda_ordinal', 'reclamacoes_por_tempo', 'taxa_resolucao', 'gasto_medio_mensal_calc', 'indicador_problemas']
Colunas categóricas identificadas: ['genero', 'estado_civil', 'tipo_contrato', 'forma_pagamento']
Dimensões do treino processado: (98872, 26)
Dimensões do teste processado: (5000, 25)

Conjuntos X e y criados.
Dimensões de X (treino): (98872, 25)
Dimensões de y (treino): (98872,)
Dimensões de X_test: (50